# Alert history: validation

Stage 3 pulled two years of daily alerts for all 3,866 territories. Before building on top of it, this notebook checks whether the data looks as expected.

It doesn’t. Two issues came up, and the second one means we need to revisit a decision made in 03_alerts.ipynb.

In [1]:
import os
import requests
from dotenv import load_dotenv
import pandas as pd
import geopandas as gpd

load_dotenv("../.env")
API_KEY = os.getenv("GFW_API_KEY")

DATASET = "gfw_integrated_alerts"
VERSION = "v20260801"
BASE = f"https://data-api.globalforestwatch.org/dataset/{DATASET}/{VERSION}/query/json"

geostores = pd.read_csv("../data/processed/geostores.csv")


def fetch(geostore_id, sql):
    r = requests.get(
        BASE,
        headers={"x-api-key": API_KEY},
        params={"sql": sql, "geostore_id": geostore_id, "geostore_origin": "rw"},
        timeout=180,
    )
    if r.status_code != 200:
        raise RuntimeError(f"{r.status_code}: {r.text[:400]}")
    return pd.DataFrame(r.json()["data"])

Row count, coverage and date range. The total is the first complete figure the project has for forest lost inside indigenous territories.

In [2]:
alerts = pd.read_csv("../data/processed/alerts_daily.csv", parse_dates=["date"])

print(f"{len(alerts):,} rows")
print(f"{alerts['territory_id'].nunique():,} territories with at least one alert")
print(f"date range: {alerts['date'].min().date()} to {alerts['date'].max().date()}")
print(f"total area lost: {alerts['area_ha'].sum():,.0f} ha")

383,397 rows
3,724 territories with at least one alert
date range: 2024-08-01 to 2026-08-07
total area lost: 3,049,683 ha


Dividing area by alert count gives the size of one alert pixel. If this stays constant, the two columns are basically telling us the same thing. The unusually high maximum alert count is worth checking.

In [3]:
print(alerts["alerts"].describe().to_string())
print()
print("area per alert:")
print(alerts["area_ha"].div(alerts["alerts"]).describe().to_string())

count    3.833970e+05
mean     6.591639e+02
std      4.388734e+04
min      1.000000e+00
25%      4.000000e+00
50%      1.400000e+01
75%      4.900000e+01
max      1.593708e+07

area per alert:
count    383397.000000
mean          0.012216
std           0.000096
min           0.011800
25%           0.012137
50%           0.012260
75%           0.012290
max           0.012310


Three million hectares in two years seems high for indigenous territories, especially since they generally have lower deforestation rates than the surrounding land.

In [4]:
territories = gpd.read_parquet("../data/processed/territories.parquet")
total_area = territories["area_ha"].sum()

print(f"total monitored area: {total_area:,.0f} ha")
print(f"total lost:           {alerts['area_ha'].sum():,.0f} ha")
print(f"share lost in 2y:     {alerts['area_ha'].sum() / total_area * 100:.2f}%")

total monitored area: 188,113,617 ha
total lost:           3,049,683 ha
share lost in 2y:     1.62%


The largest single-day figures as a share of each territory’s own area. No logging operation clears a fifth of a territory in a day, so values anywhere near that are unlikely to be real logging.

In [5]:
top = alerts.nlargest(10, "alerts").merge(
    territories[["territory_id", "name", "country", "area_ha"]],
    on="territory_id",
)
top["pct_of_territory"] = top["area_ha_x"] / top["area_ha_y"] * 100
print(top[["name", "country", "date", "alerts", "area_ha_x", "area_ha_y", "pct_of_territory"]].to_string(index=False))

               name country       date   alerts    area_ha_x    area_ha_y  pct_of_territory
            Guarayo Bolivia 2024-10-22 15937075 189619.80362 1.359088e+06         13.951986
             Kayapó  Brasil 2024-09-30 13926094 169900.04315 3.298982e+06          5.150074
            Guarayo Bolivia 2024-10-02  9143705 108754.62194 1.359088e+06          8.002028
        Monte Verde Bolivia 2024-10-24  4998367  59390.95962 9.561894e+05          6.211213
           Aripuanã  Brasil 2024-10-09  4260614  51399.99592 1.606458e+06          3.199585
            Guarayo Bolivia 2024-11-01  4110755  48911.82869 1.359088e+06          3.598871
             Kayapó  Brasil 2024-10-05  3968292  48415.27063 3.298982e+06          1.467582
            Guarayo Bolivia 2024-10-24  3546755  42117.80358 1.359088e+06          3.098975
              Baure Bolivia 2024-10-22  3512903  41944.81179 2.008194e+05         20.886835
Chácobo - Pacahuara Bolivia 2024-10-30  2926622  35273.18530 4.912537e+05       

If the outliers cluster around the same dates rather than being spread across the two years, they are more likely to be one event than a recurring pattern.

In [6]:
monthly = (
    alerts.set_index("date")
    .groupby(pd.Grouper(freq="ME"))["area_ha"]
    .sum()
    .round(0)
)
print(monthly.to_string())

date
2024-08-31     112304.0
2024-09-30     470070.0
2024-10-31    1362619.0
2024-11-30     311185.0
2024-12-31     168357.0
2025-01-31      62368.0
2025-02-28      45454.0
2025-03-31      46485.0
2025-04-30      40083.0
2025-05-31      28898.0
2025-06-30      27307.0
2025-07-31      51566.0
2025-08-31      88435.0
2025-09-30     101478.0
2025-10-31      45915.0
2025-11-30      15462.0
2025-12-31      12155.0
2026-01-31       7353.0
2026-02-28       8052.0
2026-03-31       8969.0
2026-04-30       9253.0
2026-05-31       8237.0
2026-06-30       7026.0
2026-07-31       9422.0
2026-08-31       1232.0
Freq: ME


In [7]:
territories_lite = territories[["territory_id", "country"]]
m = alerts.merge(territories_lite, on="territory_id")

peak = m[(m["date"] >= "2024-09-01") & (m["date"] <= "2024-11-30")]
print(peak.groupby("country")["area_ha"].sum().sort_values(ascending=False).round(0).to_string())

country
Brasil              1151214.0
Bolivia              915754.0
Perú                  51379.0
Guyana                 7839.0
Ecuador                7824.0
Colombia               7546.0
Venezuela              1513.0
Guyane Française        804.0


July 2026 has 9,422 ha compared with 51,566 ha in July 2025. That could be a real drop, but alerts start as `nominal` and are upgraded as more satellite passes confirm them. Since Stage 3 only keeps confirmed alerts, the most recent weeks will naturally look emptier than they really are.

In [8]:
sql_recent = (
    "SELECT gfw_integrated_alerts__date AS date, "
    "gfw_integrated_alerts__confidence AS conf, "
    "SUM(area__ha) AS area_ha "
    "FROM results "
    "WHERE gfw_integrated_alerts__date >= '2026-05-01' "
    "GROUP BY gfw_integrated_alerts__date, gfw_integrated_alerts__confidence"
)

gid = geostores.loc[geostores["territory_id"] == 4016, "geostore_id"].iloc[0]
recent = fetch(gid, sql_recent)
recent["date"] = pd.to_datetime(recent["date"])

pivot = recent.pivot_table(index=recent["date"].dt.to_period("W"),
                           columns="conf", values="area_ha", aggfunc="sum", fill_value=0)
print(pivot.tail(14).round(2).to_string())

conf                    high  highest  nominal
date                                          
2026-05-04/2026-05-10  20.18     8.50    11.09
2026-05-11/2026-05-17  14.98     1.59    17.60
2026-05-18/2026-05-24   5.44     2.79     9.45
2026-05-25/2026-05-31   2.59     0.45     5.60
2026-06-01/2026-06-07   4.76     0.38    20.22
2026-06-08/2026-06-14   8.66     0.87    11.40
2026-06-15/2026-06-21  14.35     2.90    42.69
2026-06-22/2026-06-28  13.81     0.93    27.98
2026-06-29/2026-07-05   3.06     0.44    38.89
2026-07-06/2026-07-12   3.50     1.07    17.02
2026-07-13/2026-07-19  20.81     0.81    24.55
2026-07-20/2026-07-26  23.78     1.27    49.76
2026-07-27/2026-08-02   6.20     0.00    10.32
2026-08-03/2026-08-09   1.29     0.00   221.45


The single-territory view was too noisy: with weekly volumes of 20–50 ha, one clearing event can move the whole percentage. Pooling the five largest territories gives us enough volume per week to see whether the nominal share actually falls with age.

In [9]:
pivot["nominal_pct"] = (
    pivot["nominal"] / pivot.sum(axis=1) * 100
).round(1)
print(pivot[["nominal_pct"]].tail(14).to_string())

conf                   nominal_pct
date                              
2026-05-04/2026-05-10         27.9
2026-05-11/2026-05-17         51.5
2026-05-18/2026-05-24         53.5
2026-05-25/2026-05-31         64.7
2026-06-01/2026-06-07         79.7
2026-06-08/2026-06-14         54.5
2026-06-15/2026-06-21         71.2
2026-06-22/2026-06-28         65.5
2026-06-29/2026-07-05         91.7
2026-07-06/2026-07-12         78.8
2026-07-13/2026-07-19         53.2
2026-07-20/2026-07-26         66.5
2026-07-27/2026-08-02         62.5
2026-08-03/2026-08-09         99.4


In [11]:
sql_conf_recent = (
    "SELECT gfw_integrated_alerts__date AS date, "
    "gfw_integrated_alerts__confidence AS conf, "
    "SUM(area__ha) AS area_ha "
    "FROM results "
    "WHERE gfw_integrated_alerts__date >= '2026-03-01' "
    "GROUP BY gfw_integrated_alerts__date, gfw_integrated_alerts__confidence"
)

big = [4016, 3895, 4068, 20706, 3952]

frames = []
for tid in big:
    g = geostores.loc[geostores["territory_id"] == tid, "geostore_id"].iloc[0]
    frames.append(fetch(g, sql_conf_recent))

conf_all = pd.concat(frames)
conf_all["date"] = pd.to_datetime(conf_all["date"])

wk = conf_all.pivot_table(
    index=conf_all["date"].dt.to_period("W"),
    columns="conf", values="area_ha", aggfunc="sum", fill_value=0,
)
wk["total_ha"] = wk[["nominal", "high", "highest"]].sum(axis=1).round(1)
wk["nominal_pct"] = (wk["nominal"] / wk["total_ha"] * 100).round(1)

print(wk[["total_ha", "nominal_pct"]].to_string())

conf                   total_ha  nominal_pct
date                                        
2026-02-23/2026-03-01      22.5         25.5
2026-03-02/2026-03-08     163.5          3.4
2026-03-09/2026-03-15     171.1          4.4
2026-03-16/2026-03-22     295.5         15.3
2026-03-23/2026-03-29     232.7         16.3
2026-03-30/2026-04-05     102.9         15.8
2026-04-06/2026-04-12     265.4         10.0
2026-04-13/2026-04-19     117.7          7.0
2026-04-20/2026-04-26     256.0         31.8
2026-04-27/2026-05-03     189.4         31.5
2026-05-04/2026-05-10     162.6         43.3
2026-05-11/2026-05-17     177.3         51.4
2026-05-18/2026-05-24      99.1         51.1
2026-05-25/2026-05-31     121.1         53.4
2026-06-01/2026-06-07     130.1         66.1
2026-06-08/2026-06-14     133.3         55.0
2026-06-15/2026-06-21     169.6         66.4
2026-06-22/2026-06-28     245.4         75.8
2026-06-29/2026-07-05     221.8         89.6
2026-07-06/2026-07-12     158.7         80.7
2026-07-13

## Conclusions

**Pixel size is constant** at 0.0123 ha, so `area_ha` and `alerts` contain the same information.

**142 territories recorded no alerts at all** over the two years, 3.7 per cent of the working set, mostly the smallest territories.

### Problem 1: the 2024 fires

70 per cent of the two-year loss falls within three months. October 2024 alone accounts for 1,362,619 ha, compared with 45,915 ha in October 2025. Brazil and Bolivia make up 96.5 per cent of the peak, and Baure lost 20.9 per cent of its own area in a single day. No clearing operation does that. These are burn scars being picked up as tree cover loss.

A baseline built on these two years would treat a catastrophic fire season as normal October behaviour. Genuine clearing in October 2026 could then be missed. The `wur_integration_alert_drivers_class__class` field classifies the cause of each alert and is worth testing as a way to separate fires from clearing.

### Problem 2: confidence lag makes periods incomparable

The share of alerts still marked `nominal` is 95.6 per cent in the current week, compared with 3–16 per cent by five months old. Full maturation takes around three to four months.

This reverses the filter chosen in `03_alerts.ipynb`. Filtering to `high` and `highest` captures a smaller share of recent alerts, so comparing last week with the same week a year ago is not really a comparison: the current period will always look calmer.

Stage 3 is re-run with all three confidence levels. For temporal comparisons, consistency matters more than precision. `nominal` alerts may contain false positives, but they exist in both recent and historical data. Filtering them out introduces a bias that gets larger the more recent the data is. Confidence will instead become a dashboard dimension.